# 03 · Cechy, prosty klasyfikator, dobre wzorce
**Cel:** sprawdzić, czy różnice powtarzają się na innych płytkach. Domyślnie używamy mediany cech uderzeń jednej płytki. To stabilna baza, ale może ukryć wadę słyszalną tylko w jednym miejscu — wtedy porównaj osobno stałe miejsca uderzeń.

In [1]:
from pathlib import Path
import os, json, sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Otwórz cały folder projektu w VS Code."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
from pinglab.audio import Config, load_audio, detect_hits, cut_hits, listen
from pinglab.session import load_session
SESSION = load_session(ROOT)
CONFIG, FMAX = SESSION.config, SESSION.fmax
MANIFEST, ANNOTATIONS = SESSION.manifest, SESSION.annotations
print("Dane:", SESSION.source_kind, "· manifest:", MANIFEST.relative_to(ROOT) if MANIFEST.is_relative_to(ROOT) else MANIFEST.name)
print("Wspólne pasmo do", FMAX, "Hz · konfiguracja: project.json")
if SESSION.source_kind == "synthetic":
    display(Markdown("**SYMULACJA — wynik sprawdza narzędzia, nie wykrywanie pęknięć u klienta.**"))


Dane: synthetic · manifest: data/demo/manifest.csv
Wspólne pasmo do 20000.0 Hz · konfiguracja: project.json


**SYMULACJA — wynik sprawdza narzędzia, nie wykrywanie pęknięć u klienta.**

In [2]:
# KONFIGURACJA — użyj tego samego manifestu i parametrów co w 02
MANIFEST = SESSION.manifest
ANNOTATIONS = SESSION.annotations  # dla klienta: zatwierdzony plik z 02
OUT = SESSION.output / "models"
FAMILY = 'resonance'  # resonance (3 cechy) / physics (15) / mfcc / mel
from pinglab.dataset import analyze_manifest
events, quality = analyze_manifest(MANIFEST,OUT,CONFIG,FMAX,
                                    ANNOTATIONS if ANNOTATIONS.exists() else None)

## Co oznaczają cechy?
**resonance:** tylko dominujący pik, centroid i czas zaniku — zacznij od małej liczby cech. **physics:** szerszy zestaw 15 cech, w tym pasma i płaskość. **mfcc:** zwięzły opis barwy. **mel:** kształt widma w 40 pasmach.

`decay10_s` to czas zejścia obwiedni o 10 dB w wybranym oknie, nie laboratoryjny pomiar tłumienia. Zmienny hałas i ucięty ogon mogą go zepsuć.

In [3]:
from pinglab.plots import embedding_map,distance_map
from pinglab.learning import plate_table,evaluate,anomaly_table,permutation_check
table,columns = plate_table(events,FAMILY)
display(table[['plate_id','label','n_hits','session_id']].head())
if len(table)>=3:
    display(embedding_map(events,FAMILY)[0])
    display(distance_map(events,FAMILY))

,plate_id,label,n_hits,session_id
0,SYN_001,ok,3,S1
1,SYN_002,ok,3,S2
2,SYN_003,ok,3,S3
3,SYN_004,ok,3,S1
4,SYN_005,ok,3,S2


PCA ściska wiele cech do dwóch osi. Bliskie punkty mają podobne cechy, ale „dwie chmury” nie dowodzą wykrywania defektów. Zmień kolor na sesję, by zobaczyć możliwy wpływ warunków.

In [4]:
if len(table)>=3:
    display(embedding_map(events,FAMILY,color='session_id')[0])

## Ocena klasyfikatora
Regresja logistyczna z silną regularyzacją. Każda płytka dostaje prognozę z modelu, który jej nie widział; skalowanie też uczy się tylko na treningu. Minimum programowe: 3 OK + 3 NOK; to nadal bardzo mały test. Przy mniejszej liczbie pokażemy powód zamiast sztucznej skuteczności.

In [5]:
assessment = None
try:
    assessment = evaluate(events,FAMILY)
    metrics,predictions,folds = assessment
    display(pd.Series({k:v for k,v in metrics.items() if k not in ['warning','interval_note']}))
    from pinglab.plots import prediction_plot
    display(prediction_plot(predictions))
    display(predictions)
except ValueError as error:
    print(error)

n_plates                                                                         30
n_ok                                                                             20
n_nok                                                                            10
balanced_accuracy                                                             0.975
recall_nok                                                                      1.0
false_alarm_rate                                                               0.05
roc_auc                                                                         1.0
confusion_matrix                                                 [[19, 1], [0, 10]]
recall_nok_wilson_95                       [0.7224672001371107, 0.9999999999999999]
false_alarm_wilson_95                   [0.008881448800795402, 0.23613119344674205]
wilson_note                       Opisowy przedział Wilsona dla liczby płytek; p...
threshold                                                                   

,plate_id,label,n_hits,session_id,score_nok,prediction,fold
0,SYN_001,ok,3,S1,0.199441,ok,0
1,SYN_002,ok,3,S2,0.223264,ok,4
2,SYN_003,ok,3,S3,0.377533,ok,3
3,SYN_004,ok,3,S1,0.332667,ok,0
4,SYN_005,ok,3,S2,0.230492,ok,2
5,SYN_006,ok,3,S3,0.424853,ok,1
6,SYN_007,ok,3,S1,0.290314,ok,2
7,SYN_008,ok,3,S2,0.537292,nok,0
8,SYN_009,ok,3,S3,0.187220,ok,4
9,SYN_010,ok,3,S1,0.361877,ok,4


**Recall NOK:** ile wadliwych wykryto. **Fałszywe alarmy:** ile dobrych odrzucono. Balanced accuracy uśrednia trafność obu klas. Score 0.7 nie oznacza „70% pewności wady”. Mała próba daje duży błąd oceny. Porównanie wielu metod na tych samych danych jest eksploracją; zwycięzcę sprawdź później na nowych płytkach.

In [6]:
# Ostrzejsza próba: cała nowa sesja poza treningiem.
try:
    session_metrics, session_predictions, session_folds = evaluate(events,FAMILY,by_session=True,bootstrap=0)
    display(pd.Series(session_metrics))
except ValueError as error:
    print("Test sesji:",error)

n_plates                                                                30
n_ok                                                                    20
n_nok                                                                   10
balanced_accuracy                                                     0.95
recall_nok                                                             1.0
false_alarm_rate                                                       0.1
roc_auc                                                                1.0
confusion_matrix                                        [[18, 2], [0, 10]]
recall_nok_wilson_95              [0.7224672001371107, 0.9999999999999999]
false_alarm_wilson_95           [0.027866481213768224, 0.3010336452284873]
wilson_note              Opisowy przedział Wilsona dla liczby płytek; p...
threshold                                                              0.5
family                                                           resonance
aggregation              

## Mam tylko dobre wzorce
Odległość do najbliższej dobrej płytki pomaga wskazać nietypowe nagrania. Nietypowy nie znaczy uszkodzony. Próg jest roboczy, wyliczony z niewielu dobrych wzorców; nie gwarantuje częstości fałszywych alarmów.

In [7]:
try:
    anomaly, normal_model = anomaly_table(events,FAMILY)
    display(anomaly.sort_values('distance',ascending=False))
except ValueError as error:
    print(error)

,plate_id,label,n_hits,reference,distance,exploratory_threshold,unusual
22,SYN_023,nok,3,False,3.175054,0.839503,True
27,SYN_028,nok,3,False,3.050281,0.839503,True
25,SYN_026,nok,3,False,2.736722,0.839503,True
29,SYN_030,nok,3,False,2.727940,0.839503,True
28,SYN_029,nok,3,False,2.618929,0.839503,True
23,SYN_024,nok,3,False,2.522190,0.839503,True
20,SYN_021,nok,3,False,2.451294,0.839503,True
21,SYN_022,nok,3,False,2.421461,0.839503,True
24,SYN_025,nok,3,False,2.230350,0.839503,True
26,SYN_027,nok,3,False,1.806495,0.839503,True


In [8]:
# Opcjonalnie: czy podobny wynik powstaje po przetasowaniu etykiet całych płytek?
RUN_PERMUTATION = False
if RUN_PERMUTATION:
    display(permutation_check(events,FAMILY,repeats=99))

In [9]:
from pinglab.report import write_report
source_kind=SESSION.source_kind
print(write_report(events,quality,OUT/'report.html',assessment,source_kind=source_kind))

/Users/karolsii/dev/ceramic-ping-lab/reports/runs/demo-session/models/report.html


## Zapis modelu i nowe płytki
To opcjonalny krok po sprawdzeniu oceny. Wybierz nowy katalog wersji. Model stosuje ten sam wybór cech i medianę uderzeń. Otwieraj wyłącznie własne pliki `joblib`.

In [10]:
from dataclasses import asdict
from pinglab.model import train_final,predict_new
SAVE_MODEL = False
MODEL_FOLDER = ROOT / "models/my-baseline-v1"
PROCESSING = {'config':asdict(CONFIG),'fmax':FMAX}
if SAVE_MODEL:
    print(train_final(events,MODEL_FOLDER,FAMILY,PROCESSING))
# Nowe dane (z nowymi fizycznymi płytkami) przetwórz analyze_manifest z tym samym CONFIG/FMAX:
# new_events, new_qc = analyze_manifest("data/new/manifest.csv", "reports/runs/new", CONFIG, FMAX)
# display(predict_new(new_events,MODEL_FOLDER,PROCESSING))

Dalej: **04** — gotowy model jako alternatywna reprezentacja. **05** — kontrola wpływu szumu i sposobu nagrania.